# 13 — Entrenamiento final Gradient Boosting para producción
Entrenamiento con el 100% de los datos — sin split, sin cross validation.
El modelo resultante es el que se desplegará en producción.

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import time

DELTA_COMBINED   = "/Volumes/workspace/default/network_data/features_combined/"
MODEL_OUTPUT_DIR = "/Volumes/workspace/default/network_data/modelo_produccion/"

df = spark.read.format("delta").load(DELTA_COMBINED)
print(f"Total registros : {df.count():,}")
print(f"Columnas        : {len(df.columns)}")

## 1 — Preparar todos los datos

In [0]:
exclude = [
    "window_id", "window_start", "window_end",
    "session_id", "label",
    "write_read_ratio", "min_payload_bytes",
    "max_payload_bytes", "write_read_ratio_safe",
    "lit401_fit201_ratio", "fit101_fit201_ratio", "lit301_high",
]

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in
       ("double", "long", "int", "integer", "float")
]

print(f"Features seleccionadas : {len(feature_cols)}")
print(feature_cols)

# Todos los datos — sin split
pdf = df.select(feature_cols + ["label"]).toPandas()
pdf[feature_cols] = pdf[feature_cols].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

X = pdf[feature_cols].values
y = pdf["label"].values

# Peso de clase sobre el total del dataset
n_normal      = (y == 0).sum()
n_ataque      = (y == 1).sum()
weight_ataque = round(n_normal / n_ataque, 2)

print(f"\nTotal registros : {len(X):,}")
print(f"Normal (0)      : {n_normal:,}  ({n_normal/len(y)*100:.2f}%)")
print(f"Ataque (1)      : {n_ataque:,}  ({n_ataque/len(y)*100:.2f}%)")
print(f"Peso ataque     : {weight_ataque}")

## 2 — Entrenar Gradient Boosting con todos los datos

In [0]:
# Mismos hiperparámetros que en la comparativa (NB 15)
# Se entrena con el 100% de los datos — no hay test set
# La evaluación ya se hizo en NB 15 con K-Fold CV K=5
gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

print("Entrenando Gradient Boosting con todos los datos...")
print("Este proceso puede tardar ~35 minutos...")
print("=" * 50)

t0 = time.time()
gb.fit(X, y)
t_total = round(time.time() - t0, 1)

print(f"Entrenamiento completado en {t_total}s ({t_total/60:.1f} min)")

## 3 — Verificación de correcto guardado


In [0]:
# Nota: evaluar sobre los mismos datos de entrenamiento siempre
# da métricas perfectas — esto NO es una evaluación real.
# La evaluación real ya está en NB 15 con K-Fold CV.
# Este bloque solo sirve para confirmar que el modelo aprendió correctamente.

y_pred  = gb.predict(X)
y_proba = gb.predict_proba(X)[:, 1]

cm = confusion_matrix(y, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Verificación sobre datos de entrenamiento (no es evaluación real):")
print("=" * 55)
print(classification_report(y, y_pred,
      target_names=["Normal (0)", "Ataque (1)"]))
print(f"Falsos Negativos : {fn:,}")
print(f"Falsos Positivos : {fp:,}")
print("=" * 55)
print("Evaluación real → ver NB 15 (K-Fold CV K=5)")
print(f"  AUC-ROC K-Fold : 1.0 ± 0.0001")
print(f"  Detección      : 99.08%")
print(f"  Falsa alarma   : 0.01%")

## 4 — Guardar modelo y metadatos

In [0]:
import os
import json
import pickle
from datetime import datetime

# En Databricks Serverless los volúmenes son accesibles
# directamente como rutas del sistema de ficheros local
# usando /Volumes/ sin necesidad de dbutils.fs.cp

MODEL_LOCAL_PATH = "/Volumes/workspace/default/network_data/modelo_produccion/gb_produccion.pkl"
META_LOCAL_PATH  = "/Volumes/workspace/default/network_data/modelo_produccion/gb_produccion_metadatos.json"

# Crear el directorio si no existe
os.makedirs(os.path.dirname(MODEL_LOCAL_PATH), exist_ok=True)

# Guardar modelo directamente en el volumen
with open(MODEL_LOCAL_PATH, "wb") as f:
    pickle.dump(gb, f)

print(f"Modelo guardado en : {MODEL_LOCAL_PATH}")

# Guardar metadatos
metadatos = {
    "modelo":              "GradientBoostingClassifier",
    "fecha_entrenamiento": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "n_estimators":        100,
    "max_depth":           5,
    "learning_rate":       0.1,
    "subsample":           0.8,
    "random_state":        42,
    "peso_ataque":         weight_ataque,
    "n_registros_train":   len(X),
    "n_features":          len(feature_cols),
    "features":            feature_cols,
    "umbral_optimo":       0.2,
    "metricas_kfold": {
        "auc_roc":         "1.0 ± 0.0001",
        "deteccion":       "99.08%",
        "falsa_alarma":    "0.01%",
        "fn_global":       455
    },
    "dataset":             DELTA_COMBINED,
    "ruta_modelo":         MODEL_LOCAL_PATH
}

with open(META_LOCAL_PATH, "w", encoding="utf-8") as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print(f"Metadatos guardados en : {META_LOCAL_PATH}")

# Verificar que los ficheros existen y tienen tamaño razonable
model_size = os.path.getsize(MODEL_LOCAL_PATH) / 1024 / 1024
print(f"Tamaño del modelo  : {model_size:.1f} MB")

## 5 — Resumen

In [0]:
print("=" * 55)
print("MODELO DE PRODUCCIÓN — GRADIENT BOOSTING")
print("=" * 55)
print(f"  Algoritmo          : GradientBoostingClassifier")
print(f"  Registros train    : {len(X):,}  (100% del dataset)")
print(f"  Features           : {len(feature_cols)}")
print(f"  Peso clase ataque  : {weight_ataque}")
print(f"  Tiempo entreno     : {t_total}s ({t_total/60:.1f} min)")
print("-" * 55)
print("  Métricas (K-Fold CV K=5 — NB 15):")
print(f"    AUC-ROC          : 1.0 ± 0.0001")
print(f"    Detección        : 99.08%")
print(f"    Falsa alarma     : 0.01%")
print(f"    FN               : 455")
print(f"    Umbral óptimo    : 0.2")
print("-" * 55)
print(f"  Ruta modelo        : {MODEL_OUTPUT_DIR}gb_produccion.pkl")
print("=" * 55)